In [0]:
# %sql
# DROP TABLE IF EXISTS silver.dimvertical;

In [0]:
# 1. Reads bronze.workertable.
# 2. Extracts distinct Vertical values.
# 3. Creates silver.dimvertical with auto-generated VerticalId.
# 4. Compares source verticals against existing silver.dimvertical.
# 5. Inserts only new verticals.
# 6. Lets Delta generate VerticalId automatically.

In [0]:
%run ../Misc/sharedlibraries

In [0]:
UpdatedDateTime = datetime.datetime.now()
Entity = "dimworker"

###Read Bronze tables

In [0]:
workerdf = spark.table("bronze.workertable")
verticaldf = spark.table("silver.dimvertical")

In [0]:
display(workerdf)

display(verticaldf)

###Build Dimension/Fact table

In [0]:
dimworkerdf = workerdf.filter(workerdf.RecordId.isNotNull()
    ).join(
        verticaldf,
        F.trim(workerdf.Vertical) == F.trim(verticaldf.Vertical),
        "left"
    ).select(
       workerdf.WorkerID,
       F.when(workerdf.LastProcessedChange_DateTime.isNull(), F.lit("1900-01-01")).otherwise(workerdf.LastProcessedChange_DateTime).cast("timestamp").alias("LastProcessedChange_DateTime"),
       F.from_utc_timestamp(workerdf.DataLakeModified_DateTime,'CST').alias("DataLakeModified_DateTime"),
       workerdf.SupervisorId,
       F.trim(workerdf.WorkerName).alias("WorkerName"),
       F.trim(workerdf.WorkerEmail).alias("WorkerEmail"),
       F.trim(workerdf.Phone).alias("Phone"),
       F.from_utc_timestamp(workerdf.DOJ,'CST').alias("DOJ"),
       F.from_utc_timestamp(workerdf.DOL,'CST').alias("DOL"), 
       verticaldf.VerticalId,
       F.trim(workerdf.Type).alias("Type"),
       workerdf.PayPerAnnum,
       workerdf.Rate,
       workerdf.RecordId.alias("WorkerRecordId")  
    ).withColumn("UpdatedDateTime", F.lit(UpdatedDateTime)
    ).withColumn("WorkerHashKey", F.xxhash64("WorkerRecordId")
    )

display(dimworkerdf)

###Final dataframe

In [0]:
df_final = dimworkerdf

## Write to Silver Schema

In [0]:
saveDeltaTableToCatalog(df_final, "silver", Entity)

In [0]:
print(Entity)